# AI 2002 – Assignment 2: UNO Game AI
### Adversarial Search: Minimax (Defensive) vs Expectimax (Offensive)
**GitHub Repository:** https://github.com/AhmedIftikhar-datascience/UNO-GAME-AI

## 1. Card Class & Deck Generator

In [2]:
import random
import copy

# Creates a basic UNO card with a color and a value
class Card:
    def __init__(self, color, value):
        self.color = color   # Colors like Red, Blue, Green, Yellow
        self.value = value   # Numbers 0-9 or 'Skip'

    def __repr__(self):
        return f"{self.color} {self.value}"

    def __eq__(self, other):
        return self.color == other.color and self.value == other.value


# Makes a full deck of cards and mixes them up
def generate_deck():
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    deck = []

    for color in colors:
        for number in range(10):          # Add number cards (0 to 9)
            deck.append(Card(color, number))
        deck.append(Card(color, 'Skip'))  # Add one Skip card per color

    random.shuffle(deck)  # Shuffle the deck
    return deck


# Gives 5 cards to each player
def deal_cards(deck, num_players=3, cards_each=5):
    hands = [[] for _ in range(num_players)]
    for _ in range(cards_each):
        for i in range(num_players):
            hands[i].append(deck.pop())
    return hands


print("Card class and deck generator ready.")
print("Sample card:", Card('Red', 5))
print("Sample Skip:", Card('Blue', 'Skip'))

Card class and deck generator ready.
Sample card: Red 5
Sample Skip: Blue Skip


## 2. Legal Move Generator & State Transition

In [3]:
# Finds cards that can be played on the current top card
def get_valid_moves(hand, top_card):
    valid = []
    for card in hand:
        # Match by color or value
        if card.color == top_card.color or card.value == top_card.value:
            valid.append(card)
    return valid


# Stores the current status of the game in a dictionary
def create_state(hands, top_card, deck):
    return {
        'p1_cards': hands[0],   # Player 1
        'p2_cards': hands[1],   # Player 2
        'p3_cards': hands[2],   # Player 3
        'top_card': top_card,
        'deck': deck,
        'current_player': 0,    # 0=P1, 1=P2, 2=P3
        'skip_next': False      # True if the next turn is skipped
    }


# Updates the game state after a player plays a card or draws
def apply_move(state, move, player_index):
    new_state = copy.deepcopy(state)  # Create a copy so we don't change the original state
    hand_key = ['p1_cards', 'p2_cards', 'p3_cards'][player_index]

    if move is None:
        # Draw a card (create a new deck if empty)
        if not new_state['deck']:
            new_state['deck'] = generate_deck()
        drawn = new_state['deck'].pop()
        new_state[hand_key].append(drawn)
    else:
        # Remove the played card from the hand
        new_state[hand_key] = [c for c in new_state[hand_key]
                                if not (c.color == move.color and c.value == move.value)]
        new_state['top_card'] = move

        # Handle Skip cards
        if move.value == 'Skip':
            new_state['skip_next'] = True
        else:
            new_state['skip_next'] = False

    # Move to the next player's turn
    new_state['current_player'] = (player_index + 1) % 3
    return new_state


# Checks if the game is over (someone has 0 cards)
def is_terminal(state):
    return (len(state['p1_cards']) == 0 or
            len(state['p2_cards']) == 0 or
            len(state['p3_cards']) == 0)


print("Legal move generator and state transition ready.")

Legal move generator and state transition ready.


## 3. Evaluation Function

### Formula:
Score = 50 - 5(C_{AI}) + 2(C_{opp}) + 3(S)$$

- **C_AI**: Cards in the current player's hand (fewer = better, so subtracted)
- **C_opp**: Average cards held by the two opponents (more = better for us)
- **S**: Number of Skip cards in hand (powerful for disruption)

**Weight tuning:**
- **Defensive (Minimax)**: Higher penalty on own cards (`-5`), emphasizes Skip cards (`+3`) to block opponents
- **Offensive (Expectimax)**: Higher reward on opponent cards (`+2`), encourages fast card-shedding

In [4]:
# Scores the current game based on the AI's play style (defensive or offensive)
def evaluate(state, player_index, strategy='defensive'):
    keys = ['p1_cards', 'p2_cards', 'p3_cards']
    my_hand = state[keys[player_index]]
    opp_hands = [state[keys[i]] for i in range(3) if i != player_index]

    C_AI  = len(my_hand)                                    # How many cards I have
    C_opp = sum(len(h) for h in opp_hands) / len(opp_hands) # Average cards opponents have
    S     = sum(1 for c in my_hand if c.value == 'Skip')    # How many Skip cards I have

    if strategy == 'defensive':
        # Defensive style: hate having cards, love having Skips
        w_ai, w_opp, w_skip = 5, 2, 3
    else:
        # Offensive style: care more about opponents having lots of cards
        w_ai, w_opp, w_skip = 4, 3, 2

    score = 50 - w_ai * C_AI + w_opp * C_opp + w_skip * S
    return round(score, 2)


print("Evaluation function ready.")
print("Formula: Score = 50 - w_ai*C_AI + w_opp*C_opp + w_skip*S")
print("Defensive weights: w_ai=5, w_opp=2, w_skip=3")
print("Offensive weights: w_ai=4, w_opp=3, w_skip=2")

Evaluation function ready.
Formula: Score = 50 - w_ai*C_AI + w_opp*C_opp + w_skip*S
Defensive weights: w_ai=5, w_opp=2, w_skip=3
Offensive weights: w_ai=4, w_opp=3, w_skip=2


## 4. Minimax Search (Player 1 – Defensive)

In [5]:
# Player 1 uses this defensive strategy.
# It plans ahead by assuming opponents will always try to give Player 1 the lowest score.
def minimax(state, depth, maximizing, player_index, root_player):
    # Stop searching if the game is over or we've looked far enough ahead
    if depth == 0 or is_terminal(state):
        return evaluate(state, root_player, strategy='defensive'), None

    keys = ['p1_cards', 'p2_cards', 'p3_cards']
    hand = state[keys[player_index]]
    valid_moves = get_valid_moves(hand, state['top_card'])

    # Add "draw a card" (represented by None) to the list of possible moves
    actions = valid_moves if valid_moves else []
    actions = actions + [None]  

    next_player = (player_index + 1) % 3

    if maximizing:
        # Try to get the highest possible score for our player
        best_score = float('-inf')
        best_move = None
        for move in actions:
            new_state = apply_move(state, move, player_index)
            
            # If a Skip card is played, skip the next player's turn
            if new_state['skip_next']:
                next_p = (next_player + 1) % 3
                new_state['skip_next'] = False
            else:
                next_p = next_player
                
            score, _ = minimax(new_state, depth - 1, False, next_p, root_player)
            if score > best_score:
                best_score = score
                best_move = move
        return best_score, best_move
    else:
        # Opponents try to give our player the lowest possible score
        best_score = float('inf')
        best_move = None
        for move in actions:
            new_state = apply_move(state, move, player_index)
            
            # If a Skip card is played, skip the next player's turn
            if new_state['skip_next']:
                next_p = (next_player + 1) % 3
                new_state['skip_next'] = False
            else:
                next_p = next_player
                
            # Check if it's our main player's turn next so we can go back to maximizing
            is_max = (next_p == root_player)
            score, _ = minimax(new_state, depth - 1, is_max, next_p, root_player)
            if score < best_score:
                best_score = score
                best_move = move
        return best_score, best_move


print("Minimax (Defensive) search ready. Depth = 3")

Minimax (Defensive) search ready. Depth = 3


## 5. Expectimax Search (Player 2 – Offensive)

In [6]:
# Player 2 uses this offensive strategy.
# It plans ahead by finding the best move, calculating the odds of drawing 
# helpful cards, and assuming opponents will just play a random legal card.
def expectimax(state, depth, player_index, root_player):
    # Stop searching if the game is over or we've looked far enough ahead
    if depth == 0 or is_terminal(state):
        return evaluate(state, root_player, strategy='offensive'), None

    keys = ['p1_cards', 'p2_cards', 'p3_cards']
    hand = state[keys[player_index]]
    valid_moves = get_valid_moves(hand, state['top_card'])
    next_player = (player_index + 1) % 3

    if player_index == root_player:
        # Our turn: test all valid moves to find the one with the highest score
        best_score = float('-inf')
        best_move = None

        for move in valid_moves:
            new_state = apply_move(state, move, player_index)
            next_p = (next_player + 1) % 3 if new_state['skip_next'] else next_player
            new_state['skip_next'] = False
            score, _ = expectimax(new_state, depth - 1, next_p, root_player)
            if score > best_score:
                best_score = score
                best_move = move

        # Consider the option of drawing a card
        # We calculate the likely score based on what cards are left in the deck
        if state['deck']:
            deck = state['deck']
            total = len(deck)
            expected_score = 0.0

            # Group the remaining cards to figure out the odds of drawing each type
            seen = {}
            for c in deck:
                key = (c.color, c.value)
                seen[key] = seen.get(key, 0) + 1

            for (color, value), count in seen.items():
                prob = count / total
                
                # Pretend we drew this specific card to see what happens
                sim_state = copy.deepcopy(state)
                drawn = Card(color, value)
                sim_state[keys[player_index]].append(drawn)
                sim_state['deck'] = [c for c in sim_state['deck']
                                     if not (c.color == color and c.value == value)]
                if sim_state['deck']:  
                    pass
                sim_state['current_player'] = next_player
                score, _ = expectimax(sim_state, depth - 1, next_player, root_player)
                expected_score += prob * score

            expected_score = round(expected_score, 2)

            if expected_score > best_score:
                best_score = expected_score
                best_move = None  # None means we choose to draw

        # If we have no valid moves and the deck is empty, we are forced to pass
        if not valid_moves and not state['deck']:
            return evaluate(state, root_player, 'offensive'), None

        return best_score, best_move

    else:
        # Opponent's turn: assume they just play a random legal card
        if valid_moves:
            move = random.choice(valid_moves)
        else:
            move = None  # Draw a card
            
        new_state = apply_move(state, move, player_index)
        next_p = (next_player + 1) % 3 if new_state['skip_next'] else next_player
        new_state['skip_next'] = False
        score, _ = expectimax(new_state, depth - 1, next_p, root_player)
        return score, move


print("Expectimax (Offensive) search ready. Depth = 3")
print("Nodes: MAX (AI turn) | CHANCE (draw) | OPPONENT (random legal move)")

Expectimax (Offensive) search ready. Depth = 3
Nodes: MAX (AI turn) | CHANCE (draw) | OPPONENT (random legal move)
